##SECURE SALES (RLS + CLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_fact_sales AS
SELECT
    transaction_date,
    f.store_id,
    product_id,

    -- region from dim_store
    s.sales_region,

    quantity,

    -- CLS (mask financials)
    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN total_revenue ELSE NULL END AS total_revenue,

    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN total_cost ELSE NULL END AS total_cost,

    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN total_profit ELSE NULL END AS total_profit

FROM maven_market_uc.gold.fact_sales f
LEFT JOIN maven_market_uc.gold.dim_store s
ON f.store_id = s.store_id

WHERE
    is_member('admins')
    OR is_member('data_engineers')
    OR is_member('data_analysts')
    OR is_member('business_users')
    OR (is_member('central_managers') AND s.sales_region = 'Central')
    OR (is_member('other_region_managers') AND s.sales_region != 'Central')

##SECURE RETURNS (RLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_fact_returns AS
SELECT
    r.*,
    s.sales_region
FROM maven_market_uc.gold.fact_returns r
LEFT JOIN maven_market_uc.gold.dim_store s
ON r.store_id = s.store_id

WHERE
    is_member('admins')
    OR is_member('data_engineers')
    OR is_member('data_analysts')
    OR is_member('business_users')
    OR (is_member('central_managers') AND s.sales_region = 'Central')
    OR (is_member('other_region_managers') AND s.sales_region != 'Central')

##SECURE CUSTOMER (CLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_dim_customer AS
SELECT
    customer_id,

    -- mask name
    CASE 
        WHEN is_member('admins') THEN customer_name
        ELSE 'REDACTED'
    END AS customer_name,

    customer_city,
    customer_state_province,
    customer_country,

    gender,
    marital_status,

    -- mask income
    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN yearly_income
        ELSE 'RESTRICTED'
    END AS yearly_income,

    education,
    occupation,
    homeowner

FROM maven_market_uc.gold.dim_customer

##SECURE PRODUCT (CLS)

In [0]:

CREATE OR REPLACE VIEW maven_market_uc.gold.v_secure_dim_product AS
SELECT
    product_id,
    product_name,
    product_brand,
    product_retail_price,

    CASE 
        WHEN is_member('admins') OR is_member('data_engineers')
        THEN product_cost
        ELSE NULL
    END AS product_cost,

    product_weight,
    recyclable,
    low_fat

FROM maven_market_uc.gold.dim_product

##AGG TABLES (EXECUTIVE SAFE)

In [0]:

GRANT SELECT ON maven_market_uc.gold.agg_daily_sales TO `executives`;
GRANT SELECT ON maven_market_uc.gold.agg_monthly_sales TO `executives`;
GRANT SELECT ON maven_market_uc.gold.agg_store_performance TO `executives`;
GRANT SELECT ON maven_market_uc.gold.agg_return_kpi TO `executives`;